# Notebook 05 – Tận dụng quan hệ A–B: multi-task và tinh chỉnh ngưỡng

**Ý tưởng.** A (stereotype) và B (hate speech) liên quan chặt với nhau: Cramér's V = 0,53, và 80% comment *explicit* có chứa stereotype (EDA). Vậy thay vì train hai model riêng, ta cho **một encoder dùng chung với hai head**:

```
           ┌─ head A (2 lớp) → loss A ─┐
input → encoder                         ├─ loss = loss A + loss B
           └─ head B (3 lớp) → loss B ─┘
```

Kỳ vọng: tín hiệu của A giúp B học tốt hơn và ngược lại (dữ liệu chỉ có 2989 mẫu, nên mỗi chút tín hiệu thêm đều đáng giá).

**Phần 2: tinh chỉnh cách ra quyết định.** Macro-F1 coi mọi lớp nặng như nhau, nhưng argmax của xác suất hay thiên về lớp đông. Ta cộng một độ lệch vào log-xác suất từng lớp, dò độ lệch đó theo kiểu **lồng nhau** (học trên 4 fold, chấm trên fold thứ 5) để điểm báo cáo không bị ảo. Chi tiết trong `src/sqe/decision.py`.

**Các thí nghiệm**
| Tên | Mô tả | So với |
|---|---|---|
| single | hai model riêng (đã có từ notebook 03/04) | – |
| MT | một encoder, hai head, loss = L_A + L_B | single |
| MT + cw | như MT, thêm class weight (lớp hiếm bị phạt nặng hơn) | MT |
| … + bias | tinh chỉnh ngưỡng lồng nhau trên OOF | bản không chỉnh |

In [1]:
# Chạy trên Kaggle? Ô này chép bộ dữ liệu + code (dataset bạn đã upload) sang /kaggle/working
# vì /kaggle/input chỉ đọc được, không ghi được. Trên máy nhà ô này tự bỏ qua.
import os
import shutil
from pathlib import Path

if Path("/kaggle/input").exists():
    bundle = next(Path("/kaggle/input").rglob("SemEval2027_Task8")).parent
    work = Path("/kaggle/working/sqe")
    if not work.exists():
        shutil.copytree(bundle, work)
    os.chdir(work / "SemEval2027_Task8" / "notebooks")
    print("Kaggle – thư mục làm việc:", Path.cwd())

Kaggle – thư mục làm việc: /kaggle/working/sqe/SemEval2027_Task8/notebooks


In [2]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "sqe").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from sqe.data import INPUT_MAXLEN, TASKS, load_en
from sqe.decision import nested_bias
from sqe.metrics import fmt, fold_scores, paired_delta
from sqe.results import load_oof, load_results, log_result, save_oof
from sqe.train import ENCODERS, SHORT, run_cv

pd.set_option("display.precision", 4)
df = load_en()

## 1. Chọn cấu hình nền (encoder + input) từ kết quả notebook 03–04

Multi-task cần **một** input chung cho cả A và B. Ta chọn cặp (encoder, input) có trung bình macro-F1 của A và B cao nhất, trong số các cấu hình đã chạy cho cả hai subtask.

In [3]:
res = load_results()
tr = res[res.notebook.isin(["03", "04"]) & res.model.isin(SHORT)]
pivot = tr.pivot_table(index=["model", "input"], columns="subtask", values="macro_f1").dropna()
pivot["mean_AB"] = pivot[["A", "B"]].mean(1)
pivot = pivot.sort_values("mean_AB", ascending=False)
display(pivot)
BASE_MODEL, BASE_INPUT = pivot.index[0]
short = SHORT[BASE_MODEL]
BASE_CFG = dict(ENCODERS[short], input=BASE_INPUT, max_length=INPUT_MAXLEN[BASE_INPUT])
single_name = f"{short}_{BASE_INPUT}"
print("Cấu hình nền:", BASE_CFG, "| OOF single:", single_name)

subtask                                                           A       B  \
model                                       input                             
cardiffnlp/twitter-roberta-base-hate-latest comment          0.7335  0.5870   
roberta-base                                comment          0.7274  0.5856   
                                            pair_title       0.7266  0.5808   
                                            pair_title_desc  0.7288  0.5692   
                                            concat_tdc       0.7113  0.5506   
microsoft/deberta-v3-base                   comment          0.3575  0.2134   

subtask                                                      mean_AB  
model                                       input                     
cardiffnlp/twitter-roberta-base-hate-latest comment           0.6603  
roberta-base                                comment           0.6565  
                                            pair_title        0.6537  
                                            pair_title_desc   0.6490  
                                            concat_tdc        0.6309  
microsoft/deberta-v3-base                   comment           0.2854

Cấu hình nền: {'model': 'cardiffnlp/twitter-roberta-base-hate-latest', 'input': 'comment', 'max_length': 128} | OOF single: twhate_comment


## 2. Train multi-task

In [4]:
mt = run_cv(df, f"mt_{single_name}", BASE_CFG, tasks=("A", "B"))
mtcw = run_cv(df, f"mtcw_{single_name}", dict(BASE_CFG, class_weight=True), tasks=("A", "B"))
for name, oofs, note in [(f"mt_{single_name}", mt, "multi-task"), (f"mtcw_{single_name}", mtcw, "multi-task + class weight")]:
    for t in "AB":
        log_result("05", BASE_MODEL, BASE_INPUT, t, fold_scores(oofs[t], TASKS[t][1]), note=note)

  fold 0: train 2390 | valid 599
    epoch 1: loss 1.5620 | val macro-F1 A 0.7028 | val macro-F1 B 0.4790
    epoch 2: loss 1.2178 | val macro-F1 A 0.7382 | val macro-F1 B 0.6369
    epoch 3: loss 0.9505 | val macro-F1 A 0.7452 | val macro-F1 B 0.6311
    epoch 4: loss 0.7588 | val macro-F1 A 0.7431 | val macro-F1 B 0.6111
  fold 1: train 2390 | valid 599
    epoch 1: loss 1.5622 | val macro-F1 A 0.7072 | val macro-F1 B 0.5406
    epoch 2: loss 1.2142 | val macro-F1 A 0.7347 | val macro-F1 B 0.5969
    epoch 3: loss 0.9311 | val macro-F1 A 0.7444 | val macro-F1 B 0.5840
    epoch 4: loss 0.7454 | val macro-F1 A 0.7462 | val macro-F1 B 0.6113
  fold 2: train 2392 | valid 597
    epoch 1: loss 1.5483 | val macro-F1 A 0.7143 | val macro-F1 B 0.4814
    epoch 2: loss 1.2179 | val macro-F1 A 0.7172 | val macro-F1 B 0.5337
    epoch 3: loss 0.9360 | val macro-F1 A 0.7453 | val macro-F1 B 0.5776
    epoch 4: loss 0.7558 | val macro-F1 A 0.7432 | val macro-F1 B 0.5617
  fold 3: train 2392 | va

## 3. Tinh chỉnh ngưỡng (bias) lồng nhau cho mọi biến thể

In [5]:
VARIANTS = {"single": single_name, "MT": f"mt_{single_name}", "MT + cw": f"mtcw_{single_name}"}
rows, biases_seen = [], {}
for t in "AB":
    labels = TASKS[t][1]
    base_sc = fold_scores(load_oof(single_name, t), labels)
    for label, name in VARIANTS.items():
        oof = load_oof(name, t)
        tuned, biases = nested_bias(oof, labels)
        save_oof(tuned, f"{name}_bias", t)
        biases_seen[(t, label)] = biases
        for suffix, o in [("", oof), (" + bias", tuned)]:
            sc = fold_scores(o, labels)
            d = paired_delta(sc, base_sc)
            rows.append({"subtask": t, "biến thể": label + suffix, "macro_f1": sc.macro_f1.mean(),
                         "std": sc.macro_f1.std(ddof=0),
                         "Δ so với single": d["delta_mean"], "thắng single": f"{d['wins']}/5",
                         **{c: sc[c].mean() for c in sc.columns if c.startswith("f1_")}})
        log_result("05", BASE_MODEL, BASE_INPUT, t, fold_scores(tuned, labels), note=f"{label} + bias lồng nhau")
table = pd.DataFrame(rows)
table

,subtask,biến thể,macro_f1,std,Δ so với single,thắng single,f1_no,f1_yes,f1_yes_implicit,f1_yes_explicit
0,A,single,0.7335,0.0115,0.0000,0/5,0.7649,0.7020,NaN,NaN
1,A,single + bias,0.7311,0.0115,-0.0024,0/5,0.7644,0.6977,NaN,NaN
2,A,MT,0.7354,0.0111,0.0019,4/5,0.7653,0.7055,NaN,NaN
3,A,MT + bias,0.7362,0.0110,0.0027,4/5,0.7708,0.7016,NaN,NaN
4,A,MT + cw,0.7301,0.0127,-0.0034,2/5,0.7556,0.7045,NaN,NaN
5,A,MT + cw + bias,0.7352,0.0109,0.0017,4/5,0.7703,0.7002,NaN,NaN
6,B,single,0.5870,0.0114,0.0000,0/5,0.7364,NaN,0.4246,0.6001
7,B,single + bias,0.5905,0.0108,0.0034,3/5,0.7268,NaN,0.4403,0.6043
8,B,MT,0.6006,0.0259,0.0136,4/5,0.7501,NaN,0.4313,0.6204
9,B,MT + bias,0.5962,0.0160,0.0092,4/5,0.7339,NaN,0.4412,0.6136


**Bias học được ở từng fold** (ví dụ cho bản single). Nếu bias ổn định giữa các fold thì việc tinh chỉnh đáng tin; nếu dao động mạnh thì nó chủ yếu đang khớp theo nhiễu.

In [6]:
for t in "AB":
    labels = TASKS[t][1]
    b = pd.DataFrame(biases_seen[(t, "single")], index=[f"b_{l}" for l in labels]).T
    b.index.name = "fold"
    print(f"Subtask {t}:")
    display(b)

Subtask A:


,b_no,b_yes
fold,,
0,0.0,-0.1
1,0.0,0.0
2,0.0,-0.1
3,0.0,0.0
4,0.0,-0.2


Subtask B:


,b_no,b_yes_implicit,b_yes_explicit
fold,,,
0,0.0,0.6,1.1
1,0.0,0.6,1.1
2,0.0,0.9,0.3
3,0.0,0.6,1.0
4,0.0,0.6,1.1


## 4. Model đa nhiệm có "hiểu" quan hệ A–B không?

So tỉ lệ P(A = yes | B) trong nhãn thật và trong dự đoán của hai cách (single, MT). Nếu multi-task học được mối liên hệ, tỉ lệ ở dự đoán MT sẽ gần nhãn thật hơn.

In [7]:
def joint(name_A, name_B):
    a, b = load_oof(name_A, "A"), load_oof(name_B, "B")
    m = a[["StereoQueerEval_id", "y_pred"]].merge(b[["StereoQueerEval_id", "y_pred"]], on="StereoQueerEval_id",
                                                  suffixes=("_A", "_B"))
    return m.groupby("y_pred_B")["y_pred_A"].apply(lambda s: (s == "yes").mean())


truth = df.groupby("hate_speech")["stereotype"].apply(lambda s: (s == "yes").mean())
pd.DataFrame({"nhãn thật": truth, "single": joint(single_name, single_name),
              "MT": joint(f"mt_{single_name}", f"mt_{single_name}")}).rename_axis("B").round(3)

,nhãn thật,single,MT
B,,,
no,0.185,0.186,0.167
yes_explicit,0.802,0.855,0.924
yes_implicit,0.539,0.498,0.472


## 5. Kết luận

*(phần này được viết sau khi chạy xong, dựa trên số liệu ở trên)*